# 21. A goalie prior, and is goalie WAR on the same scale?

Forecast, Phase 2, step 1. Notebook 15 pulls every goalie toward **the league-average goalie** with 12,000 minutes of weight (about four seasons of a starter), because goalie results barely repeat. That pull is the right size, but it pulls **everyone toward the same place**: a 39-year-old backup and a young starter with a great track record both lean toward average. In notebook 20's first version all 72 regular goalies rated 81-95, and 24% were 90+ (7% of forwards).

**Part 1, the goalie prior:** like notebook 18 did for skaters, each goalie is pulled toward **his own expected level**, built from a few things we know before the season:

- **NHL experience:** prior seasons with 500+ minutes (prorated for short seasons), capped at 7,
- **age**,
- **career NHL minutes** before the season (log scale: the first 1,000 minutes say more than minutes 10,000-11,000),
- **last season's minutes** (his role),
- **pre-NHL save%:** how much better he stopped pucks than his league's average before he reached the NHL (AHL, NCAA, Europe, juniors).

Samples are small (about 30-60 regular goalies a season), so the model keeps few inputs. Safeguards:

- pre-NHL save% is **shrunk toward his league's average by how many shots it's based on** (a 35-game junior season says less than a full AHL career),
- **age is never extrapolated** below the youngest goalie in the fitting data,
- **goalie prospects don't get an age bonus they haven't earned** (section 8): they're rated at least at the median age of a goalie's first 500+ minute NHL season.

**Part 2, the scale diagnosis:** a read-only check before this step showed the prior alone won't remove the cluster, because even in **actual** results the average NHL backup is well above replacement. So two tests ask whether goalie WAR is on the same scale as skater WAR, and the notebook ends with a plain verdict and options.

**OVR in this notebook** uses `ratings.py` as it is now: after this notebook was first run, Ryan gave goalies their own OVR anchors (notebook 20, second version: median regular goalie = 80, top goalie = 95). So all OVR numbers here are on the goalie scale; the projections themselves don't depend on it.

The code is in `src/development.py` (section 12). Results stay private (`data/goalie_projections_2026-27.csv`, gitignored). `projections_2026-27.csv` (notebook 18) is not touched.

## 1. Setup and inputs

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, add_age, GOALS_PER_WIN, SHORT_SEASONS
from edge_data import build_bio_table
from nhle import load_profiles
from development import (load_goalie_seasons, first_nhl_goalie_season, relative_save_pct, save_pct_repeat_k,
                         shrink_save_pct, pre_nhl_save_pct, goalie_candidates, goalie_features, goalie_fit_rows,
                         fit_prior, apply_prior, model_table, wls, wmean, TARGETS, AGING, G_WEIGHTS, G_REG,
                         GOALIE_VERSIONS)
from ratings import war_std, ovr, ovr_exact, STD_MINUTES, GOALIE_ANCHOR_WAR

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where goalie_projections_2026-27.csv is saved

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
goalies = history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
gseasons = load_goalie_seasons(DATA / "bio_raw", DATA / "prospect_raw")      # NHL goalies and goalie prospects
proj18 = pd.read_csv(DATA / "projections_2026-27.csv")                       # notebook 18 (goalies unchanged there)
pros = pd.read_csv(DATA / "prospects_2026-27.csv")                           # notebook 19

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
print(f"Goalies in the WAR history: {goalies['playerId'].nunique()}; goalie profiles read: {gseasons['playerId'].nunique()}")
print("Train targets:", ", ".join(label(t) for t in TRAIN))
print("Test targets: ", ", ".join(label(t) for t in TEST))

Goalies in the WAR history: 341; goalie profiles read: 500
Train targets: 2009-10, 2010-11, 2011-12, 2013-14, 2014-15, 2015-16, 2016-17, 2017-18
Test targets:  2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


**Pre-NHL save%.** Profiles list save% for most leagues. A raw .915 means little on its own (some leagues score more), so each season is measured **against that league-season's average** (shots-weighted, over the goalies in our profiles with 10+ games, league-seasons with 3+ of them). As with skaters' NHLe (notebook 19), a goalie's pre-NHL record is the games-weighted average of his **last two seasons** before his first NHL season with 500+ minutes. +1.0 means one point of save% better than his league (.911 where the league is .901).

It's measured **as known going into each season**: only seasons before the target season count. Profiles already list 2026-27 games (the season is underway), and a backtest must never see seasons after the one it predicts.

**Shrinking by sample size.** Save% over a few hundred shots is mostly luck. So the raw relative save% is pulled toward 0 (his league's average) by the shots it's based on: **pre_sv = raw x shots / (shots + K)**. K is the number of shots at which a record counts half. It's estimated from the data, not from the test seasons: how much relative save% **repeats** between a goalie's consecutive pre-NHL seasons. Luck doesn't repeat, so the covariance between the two seasons is the spread of true talent; a season of n shots adds luck with variance p(1 - p) / n (p = league save%), which gives K = p(1 - p) / (true talent variance). A backtest of K = 1,000, 2,000 and 4,000 shots in section 3 checks it. Shots faced come from the profiles; where a league lists save% but no shots, they're games x that league-season's average shots per game.

In [ ]:
pre = gseasons[(gseasons["league"] != "NHL") & (gseasons["gp"] >= 10)]
cov = pre.groupby("league").agg(seasons=("gp", "size"), with_save_pct=("save_pct", lambda x: x.notna().mean()))
print("Save% coverage, pre-NHL seasons with 10+ games (most common leagues):")
print(cov.sort_values("seasons", ascending=False).head(12).round(2).to_string())

first_nhl = first_nhl_goalie_season(history, gseasons)
rel = relative_save_pct(gseasons)
pre_rows = rel[rel["season"] < rel["playerId"].map(first_nhl).fillna(np.inf)]
print(f"\nPre-NHL seasons with a save% (10+ games): {len(pre_rows):,}; shots faced filled in as games x league "
      f"shots per game for {pre_rows['shots_imputed'].sum()} ({pre_rows['shots_imputed'].mean():.1%})")

rep = save_pct_repeat_k(rel, first_nhl)
K_EST = rep["K"]
K = int(round(K_EST / 500) * 500)
print(f"Consecutive pre-NHL seasons: {rep['pairs']:,} pairs, correlation {rep['corr']:.2f} (median season {rep['median_season_shots']:,.0f} shots)")
print(f"Spread of true talent: {rep['true_sd_points']:.2f} points of save% (league save% {rep['league_save_pct']:.3f})")
print(f"K = p(1 - p) / true variance = {K_EST:,.0f} shots, rounded to {K:,}")
print("Share of a record that counts, shots / (shots + K): " + ", ".join(f"{n:,} shots {n / (n + K):.0%}" for n in [500, 1000, 2000, 4000]))

pre_sv = pre_nhl_save_pct(gseasons, first_nhl, before=2026, shrink_shots=K)     # as known going into 2026-27
print(f"\nGoalies with a pre-NHL save% going into 2026-27: {len(pre_sv)}; median {pre_sv['pre_sv_shots'].median():,.0f} shots in his last two seasons")
print(f"SD raw {pre_sv['pre_sv_raw'].std():.2f}, shrunk {pre_sv['pre_sv'].std():.2f} points of save%")

# Features for every goalie the projection covers (targets and 2026-27), plus every goalie-season used for fitting
pairs = pd.concat([goalie_candidates(history, TARGETS + [2026]),
                   goalies[goalies["season"].isin(TARGETS)][["playerId", "season"]].assign(group="G")]).drop_duplicates(["playerId", "season"])
feat = goalie_features(pairs, history, bios, profiles, gseasons, first_nhl, shrink_shots=K)
print(f"Goalie-seasons with features: {len(feat):,}; pre-NHL save% known for {feat['pre_sv'].notna().mean():.0%}")

# Age floor: never extrapolate below the youngest goalie the prior is fit on
AGE_FLOOR = int(goalie_fit_rows(history, feat, TARGETS)["age"].min())
feat["age_raw"] = feat["age"]
feat["age"] = feat["age"].clip(lower=AGE_FLOOR)
print(f"Youngest age in the prior's fitting data: {AGE_FLOOR}. Ages below it count as {AGE_FLOOR} "
      f"({(feat['age_raw'] < AGE_FLOOR).sum()} NHL goalie-seasons affected; prospects in section 8)")

# Fading versions of pre-NHL save% (section 3): only for 3 or fewer NHL seasons, or fading to 0 at 7 seasons
def add_fades(d):
    d = d.copy()
    d["pre_sv_young"] = d["pre_sv"].where(d["experience"] <= 3, 0.0)
    d["pre_sv_fade"] = d["pre_sv"] * (1 - d["experience"] / 7).clip(lower=0)
    return d
feat = add_fades(feat)

Save% coverage, pre-NHL seasons with 10+ games (most common leagues):
         seasons  with_save_pct
league                         
AHL         1314           0.97
ECHL         303           0.96
WHL          225           0.93
OHL          223           0.79
KHL          206           0.87
QMJHL        173           0.73
NCAA         157           1.00
USHL         126           0.83
MHL          105           1.00
SHL          104           1.00
Liiga         98           1.00
Finland       80           0.00

Pre-NHL seasons with a save% (10+ games): 2,823; shots faced filled in as games x league shots per game for 93 (3.3%)
Consecutive pre-NHL seasons: 1,888 pairs, correlation 0.19 (median season 910 shots)
Spread of true talent: 0.56 points of save% (league save% 0.910)
K = p(1 - p) / true variance = 2,591 shots, rounded to 2,500
Share of a record that counts, shots / (shots + K): 500 shots 17%, 1,000 shots 29%, 2,000 shots 44%, 4,000 shots 62%

Goalies with a pre-NHL save% going

In [ ]:
# Check: without priors, the projection is notebook 18's saved goalie projection (notebook 15's method)
curve_full = aging_curve(history, bios)[1]
old_g = project(goalies, bios, 2026, G_WEIGHTS, G_REG, AGING, curve_full)
saved = proj18[proj18["position"] == "G"].set_index("playerId")["new_proj_war_60"]
diff = (old_g["proj_war_60"].reindex(saved.index) - saved).abs().max()
assert len(old_g) == len(saved) and diff < 1e-9, diff
print(f"No-prior goalie projection matches projections_2026-27.csv ({len(saved)} goalies, largest difference {diff:.1e})")

No-prior goalie projection matches projections_2026-27.csv (132 goalies, largest difference 9.7e-17)


## 2. The prior model

For each version, a minutes-weighted regression of **actual GAR/60** on the inputs, using every goalie-season with 500+ minutes in the training target seasons (2009-10 to 2017-18, rookies included).

| Version | Inputs |
|---|---|
| a | experience, age |
| b | (a) + career minutes (log) |
| c | (b) + last season's minutes |
| d | the best of (a)-(c) + pre-NHL save% (shrunk) |
| e1 | the best of (a)-(c) + pre-NHL save% only for goalies with 3 or fewer NHL seasons |
| e2 | the best of (a)-(c) + pre-NHL save% fading with experience (full at 0 seasons, half at 3.5, none at 7) |

(e1) and (e2) test whether junior and AHL numbers stop mattering once a goalie has an NHL record.

**Expected signs:** more experience, more career minutes, more minutes last season and a better pre-NHL save%: higher. **Age:** at the same experience, an older goalie either reached the NHL late (usually a lesser prospect) or is past his peak, so age should be negative.

In [ ]:
fit_train = goalie_fit_rows(history, feat, TRAIN)
models = {v: fit_prior(fit_train, cols, fill_col="pre_sv") for v, cols in GOALIE_VERSIONS.items()}
EXPECT = {"experience": +1, "age": -1, "log_career": +1, "last_k": +1, "pre_sv": +1, "pre_sv_young": +1, "pre_sv_fade": +1}
FILL = {"d": "pre_sv", "e1": "pre_sv_young", "e2": "pre_sv_fade"}      # the input that can be missing, per version

def sign_table(models):
    rows = []
    for v, m in models.items():
        m = m["G"]
        for k in m["inputs"]:
            c, s = m["coef"][k], m["se"][k]
            ok = np.sign(c) == EXPECT[k]
            rows.append({"version": v, "input": k, "coef": c, "se": s, "z": c / s,
                         "sign": "as expected" if ok else ("WRONG (beyond noise)" if abs(c / s) > 2 else "wrong, within noise")})
    return pd.DataFrame(rows)

print(f"Fit on {len(fit_train)} goalie-seasons (500+ minutes, training targets). Weighted R^2: "
      + ", ".join(f"({v}) {m['G']['r2']:.3f}" for v, m in models.items()))
sign_table(models).round(4)

Fit on 527 goalie-seasons (500+ minutes, training targets). Weighted R^2: (a) 0.030, (b) 0.034, (c) 0.035


,version,input,coef,se,z,sign
0,a,experience,0.0286,0.0081,3.5385,as expected
1,a,age,-0.0186,0.0048,-3.9052,as expected
2,b,experience,0.0059,0.0163,0.3643,as expected
3,b,age,-0.0199,0.0048,-4.1163,as expected
4,b,log_career,0.0608,0.0379,1.6041,as expected
5,c,experience,0.0090,0.0174,0.5164,as expected
6,c,age,-0.0191,0.0051,-3.7748,as expected
7,c,log_career,0.0457,0.0482,0.9484,as expected
8,c,last_k,0.0076,0.0151,0.5037,as expected


## 3. Backtest

Same setup as notebook 18: priors fit on target seasons through 2017-18, tested on 2018-19 onward. The tested goalies are notebook 15's: **1,000+ minutes in the target season and a season the year before** (so true rookies are tested separately in section 4). A check on goalies with **500+ minutes** (more backups) follows.

Each version at 6,000, 12,000 and 24,000 minutes of prior weight. Error = minutes-weighted RMSE of GAR/60, for all goalies, goalies with 3 or fewer prior NHL seasons, and veterans (4+).

**Picking rule:** the lowest all-goalie test error among (a)-(c), veterans not worse than notebook 15. Then (d), (e1) and (e2) add pre-NHL save% to the winner in their three ways, and one is kept only if it lowers the all-goalie test error (veterans not worse). A fading version has to beat (d) by **more than 0.001** to replace it: (d) is the simpler model, and smaller gaps are noise (the same threshold as the weight tie rule below). **Settings are picked on the test seasons**, so the winner's error is a little optimistic; the training ranking (prior fit on the same seasons it's scored on) is shown next to it.

**Tie rule for the weight:** if notebook 15's 12,000 minutes is within 0.001 of the best weight for the chosen version, 12,000 is used. Differences that small are noise (checked below, season by season and by resampling), and 12,000 keeps notebook 15's goalie weight, the same principle notebook 18 followed for skaters (1,500 minutes). It also gives a goalie's own record more say than a larger weight would.

In [ ]:
info = feat[["playerId", "season", "experience"]]
cand_test = goalie_candidates(history, TEST).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
cand_train = goalie_candidates(history, TRAIN).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")

def prior_rows(cand, model):
    return cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())

def run(targets, priors=None, reg=None, min_toi=1000):
    b = backtest(goalies, targets, G_WEIGHTS, G_REG, min_toi, bios, aging=AGING, priors=priors, prior_reg=reg)
    return b.merge(info, on=["playerId", "season"], how="left")

def score(b):
    rm = lambda m: np.sqrt(np.average((b["actual"][m] - b["pred"][m]) ** 2, weights=b["toi"][m]))
    return {"goalies": len(b), "all": rm(b["toi"] > 0), "3 or fewer seasons": rm(b["experience"] <= 3),
            "veterans (4+)": rm(b["experience"] >= 4),
            "bias (pred - actual, WAR/60)": np.average(b["pred"] - b["actual"], weights=b["toi"]) / GOALS_PER_WIN}

WEIGHTS_G = [6000, 12000, 24000]
def grid_for(model_set, versions, min_toi=1000):
    rows = []
    for period, tg, cand in [("test", TEST, cand_test), ("train", TRAIN, cand_train)]:
        rows.append({"period": period, "version": "notebook 15", "weight": G_REG, **score(run(tg, min_toi=min_toi))})
        for v in versions:
            pr = prior_rows(cand, model_set[v])
            for w in WEIGHTS_G:
                rows.append({"period": period, "version": v, "weight": w, **score(run(tg, pr, w, min_toi))})
    return pd.DataFrame(rows)

def pick(grid, versions):
    t = grid[grid["period"] == "test"].set_index(["version", "weight"])
    base = t.loc[("notebook 15", G_REG), "veterans (4+)"]
    ok = t[t.index.get_level_values(0).isin(versions) & (t["veterans (4+)"] <= base + 1e-12)]
    return ok["all"].idxmin() if len(ok) else None

g1 = grid_for(models, list(GOALIE_VERSIONS))
best_abc = pick(g1, list(GOALIE_VERSIONS))
print("Test seasons (2018-19 on), goalies with 1,000+ minutes:")
print(g1[g1["period"] == "test"].drop(columns="period").round(4).to_string(index=False))
print(f"\nBest of (a)-(c): version {best_abc[0]}, {best_abc[1]:,} minutes" if best_abc else "\nNo version beats notebook 15 without hurting veterans.")

Test seasons (2018-19 on), goalies with 1,000+ minutes:
    version  weight  goalies    all  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)
notebook 15   12000      370 0.3043              0.3177         0.2963                       -0.0011
          a    6000      370 0.3059              0.3188         0.2982                        0.0007
          a   12000      370 0.3028              0.3167         0.2945                       -0.0003
          a   24000      370 0.3030              0.3171         0.2946                       -0.0013
          b    6000      370 0.3063              0.3194         0.2986                        0.0001
          b   12000      370 0.3033              0.3175         0.2948                       -0.0010
          b   24000      370 0.3035              0.3180         0.2949                       -0.0020
          c    6000      370 0.3066              0.3195         0.2990                        0.0001
          c   12000      370 0.3036

In [ ]:
# Check K: version (d) at notebook 15's 12,000 minutes, with pre-NHL save% raw or shrunk with different K
base_v = best_abc[0] if best_abc else "a"
D_INPUTS = GOALIE_VERSIONS[base_v] + ["pre_sv"]
def with_k(d, k):
    d = d.copy()
    d["pre_sv"] = d["pre_sv_raw"] if k is None else shrink_save_pct(d["pre_sv_raw"], d["pre_sv_shots"], k)
    return d
krows = []
for k in dict.fromkeys([None, 1000, 2000, K, 4000]):
    m = fit_prior(with_k(fit_train, k), D_INPUTS, fill_col="pre_sv")["G"]
    b = run(TEST, prior_rows(with_k(cand_test, k), {"G": m}), G_REG)
    krows.append({"K (shots)": "none (raw)" if k is None else f"{k:,}" + (" (estimated)" if k == K else ""),
                  "pre_sv coef": m["coef"]["pre_sv"], "z": m["coef"]["pre_sv"] / m["se"]["pre_sv"], **score(b)})
print(f"Version (d) at 12,000 minutes, test seasons, by shrinkage K:")
print(pd.DataFrame(krows).round(4).to_string(index=False))
print(f"Used: K = {K:,} shots (estimated from how save% repeats, not picked on the test seasons)")

Version (d) at 12,000 minutes, test seasons, by shrinkage K:
        K (shots)  pre_sv coef      z  goalies    all  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)
       none (raw)       0.0496 2.4258      370 0.3011              0.3134         0.2938                        0.0014
            1,000       0.0780 2.5024      370 0.3013              0.3133         0.2942                        0.0012
            2,000       0.1034 2.5179      370 0.3014              0.3134         0.2944                        0.0012
2,500 (estimated)       0.1158 2.5205      370 0.3015              0.3134         0.2944                        0.0011
            4,000       0.1521 2.5220      370 0.3016              0.3134         0.2945                        0.0011
Used: K = 2,500 shots (estimated from how save% repeats, not picked on the test seasons)


In [ ]:
# (d), (e1), (e2): the best of (a)-(c) plus pre-NHL save% (shrunk with K), in three ways
for v in ["d", "e1", "e2"]:
    models[v] = fit_prior(fit_train, GOALIE_VERSIONS[base_v] + [FILL[v]], fill_col=FILL[v])
print(f"Versions (d), (e1), (e2) = ({base_v}) + pre-NHL save%:")
print(sign_table({v: models[v] for v in ["d", "e1", "e2"]}).round(4).to_string(index=False))

g2 = grid_for(models, ["d", "e1", "e2"])
grid = pd.concat([g1, g2[g2["version"].isin(["d", "e1", "e2"])]], ignore_index=True)
print("\nTest seasons, all versions:")
print(grid[grid["period"] == "test"].drop(columns="period").round(4).to_string(index=False))
tr = grid[grid["period"] == "train"].drop(columns="period")
print("\nTraining seasons (prior fit on these same seasons, so in-sample):")
print(tr.assign(rank=tr["all"].rank().astype(int)).round(4).to_string(index=False))

best_d = pick(grid, ["d"])
best_e = pick(grid, ["e1", "e2"])
t_all = grid[grid["period"] == "test"].set_index(["version", "weight"])["all"]
candidates_v = list(GOALIE_VERSIONS) + ["d"]
if best_e is not None and best_d is not None:
    gap = t_all[best_d] - t_all[best_e]
    fading_wins = gap > 0.001
    print(f"\nBest fading version: ({best_e[0]}) {best_e[1]:,} minutes {t_all[best_e]:.4f} vs. (d) {best_d[1]:,} minutes {t_all[best_d]:.4f} "
          f"(gap {gap:+.4f}): " + ("fading beats (d) by more than 0.001, so it competes"
                                   if fading_wins else "not more than 0.001 better, so (d), the simpler model, is kept"))
    if fading_wins:
        candidates_v += ["e1", "e2"]
choice = pick(grid, candidates_v)
if choice is None:
    V, W, W_LOW = None, None, None
    print("\nChosen: notebook 15 (no goalie prior beats it without hurting veterans)")
else:
    V, W_LOW = choice
    t = grid[grid["period"] == "test"].set_index(["version", "weight"])
    tie = abs(t.loc[(V, G_REG), "all"] - t.loc[(V, W_LOW), "all"]) <= 0.001 and t.loc[(V, G_REG), "veterans (4+)"] <= t.loc[("notebook 15", G_REG), "veterans (4+)"]
    W = G_REG if tie else W_LOW
    print(f"\nLowest all-goalie test error, veterans not worse: version {V} ({', '.join(models[V]['G']['inputs'])}), {W_LOW:,} minutes "
          f"({t.loc[(V, W_LOW), 'all']:.4f})")
    if W != W_LOW:
        print(f"Tie rule: 12,000 minutes is within 0.001 ({t.loc[(V, G_REG), 'all']:.4f}), so the chosen weight is 12,000 minutes")
    print(f"Chosen: version {V}, {W:,} minutes")
    tri = tr.set_index(["version", "weight"])
    tv = tri[tri.index.get_level_values(0) != "notebook 15"]["all"].idxmin()
    print(f"Best on the training seasons (in-sample): version {tv[0]}, {tv[1]:,} minutes")

Versions (d), (e1), (e2) = (a) + pre-NHL save%:
version        input    coef     se       z        sign
      d   experience  0.0288 0.0080  3.5732 as expected
      d          age -0.0184 0.0047 -3.8816 as expected
      d       pre_sv  0.1158 0.0459  2.5205 as expected
     e1   experience  0.0307 0.0082  3.7604 as expected
     e1          age -0.0181 0.0048 -3.8026 as expected
     e1 pre_sv_young  0.1239 0.0739  1.6764 as expected
     e2   experience  0.0319 0.0081  3.9271 as expected
     e2          age -0.0179 0.0047 -3.7701 as expected
     e2  pre_sv_fade  0.2455 0.0867  2.8309 as expected

Test seasons, all versions:
    version  weight  goalies    all  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)
notebook 15   12000      370 0.3043              0.3177         0.2963                       -0.0011
          a    6000      370 0.3059              0.3188         0.2982                        0.0007
          a   12000      370 0.3028              0.3167     

In [ ]:
# Check on goalies with 500+ minutes in the target season (more backups), same settings
rows = [{"version": "notebook 15", "weight": G_REG, **score(run(TEST, min_toi=500))}]
for v in list(GOALIE_VERSIONS) + ["d", "e1", "e2"]:
    rows.append({"version": v, "weight": W or G_REG, **score(run(TEST, prior_rows(cand_test, models[v]), W or G_REG, 500))})
print(f"Test seasons, goalies with 500+ minutes, {W or G_REG:,} minutes of prior weight:")
print(pd.DataFrame(rows).round(4).to_string(index=False))

Test seasons, goalies with 500+ minutes, 12,000 minutes of prior weight:
    version  weight  goalies    all  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)
notebook 15   12000      412 0.3196              0.3425         0.3052                        0.0004
          a   12000      412 0.3177              0.3407         0.3032                        0.0010
          b   12000      412 0.3180              0.3411         0.3033                        0.0003
          c   12000      412 0.3182              0.3411         0.3037                        0.0003
          d   12000      412 0.3169              0.3391         0.3030                        0.0025
         e1   12000      412 0.3172              0.3393         0.3032                        0.0020
         e2   12000      412 0.3176              0.3419         0.3023                        0.0027


## 4. True rookie goalies

A goalie in his first real NHL season has no NHL history to blend, so his projection **is** his prior. Test: every goalie whose **first NHL season with 500+ minutes** is a test season (2018-19 on). Some had a few games before (under 500 minutes), which is normal for goalies. Three candidates:

- the **chosen prior alone** (fit through 2017-18),
- the **league-average goalie** (notebook 15, and what goalie prospects got in notebook 19),
- **replacement level** (WAR/60 = 0, notebook 20's placeholder).

The winner decides what goalie prospects get.

In [ ]:
gtest = goalies[goalies["season"].isin(TEST) & (goalies["toi"] >= 500)]
rk = gtest[gtest["season"] == gtest["playerId"].map(first_nhl)].copy()
rk["gar_60"] = rk["gar"] / rk["toi"] * 60
rk = rk.merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
earlier = rk["career_min"] > 0
print(f"True rookie goalies (first 500+ minute season 2018-19 on): {len(rk)} "
      f"({earlier.sum()} had some NHL minutes before, median {rk.loc[earlier, 'career_min'].median():.0f})")
rook_model = models[V] if V else models["a"]
cands = {f"prior ({V or 'a'})": apply_prior(rk, rook_model),
         "league-average goalie": pd.Series([group_average(history, s - 1)["G"] for s in rk["season"]], index=rk.index),
         "replacement (0)": pd.Series(0.0, index=rk.index)}
rrows = []
for name, pred in cands.items():
    e = pred - rk["gar_60"]
    rrows.append({"projection": name, "RMSE (GAR/60)": np.sqrt(np.average(e ** 2, weights=rk["toi"])),
                  "bias (pred - actual, WAR/60)": np.average(e, weights=rk["toi"]) / GOALS_PER_WIN,
                  "mean projection (WAR/60)": np.average(pred, weights=rk["toi"]) / GOALS_PER_WIN})
rookie_table = pd.DataFrame(rrows).set_index("projection")
print(f"Their actual WAR/60 (minutes-weighted): {np.average(rk['gar_60'], weights=rk['toi']) / GOALS_PER_WIN:.4f}")
ROOKIE_RULE = rookie_table["RMSE (GAR/60)"].idxmin()
print(f"Lowest error: {ROOKIE_RULE}")
rookie_table.round(4)

True rookie goalies (first 500+ minute season 2018-19 on): 54 (40 had some NHL minutes before, median 207)
Their actual WAR/60 (minutes-weighted): 0.0405
Lowest error: prior (d)


,RMSE (GAR/60),"bias (pred - actual, WAR/60)",mean projection (WAR/60)
projection,,,
prior (d),0.4398,0.0031,0.0436
league-average goalie,0.4440,0.0054,0.0459
replacement (0),0.5051,-0.0405,0.0000


## 5. Feedback check for the simulator

If last season's minutes is an input, a goalie who gets more starts one season gets a higher prior the next, which can feed on itself in a multi-year simulation (more starts, higher rating, more starts). This measures how big that loop is: the prior's change when last season's minutes go from a backup's 1,500 to a starter's 3,000, in WAR/60 and in OVR at a typical goalie projection.

In [ ]:
if V and "last_k" in models[V]["G"]["inputs"]:
    k = models[V]["G"]["coef"]["last_k"]
    rise_gar = k * (3.0 - 1.5)                      # last_k is in thousands of minutes
    rise_war = rise_gar / GOALS_PER_WIN
    typical = np.average(old_g["proj_war_60"], weights=old_g["minutes_used"])
    print(f"Prior rise from 1,500 to 3,000 minutes last season: {rise_gar:+.4f} GAR/60 = {rise_war:+.4f} WAR/60")
    print(f"The projection moves by about {W / (W + 1500 + 3000):.0%} of the prior's change for a goalie with 4,500 recent minutes: "
          f"OVR {ovr(typical, 'G')} -> {ovr(typical + rise_war * W / (W + 4500), 'G')} at a typical projection ({typical:.3f})")
else:
    print("Last season's minutes isn't in the chosen prior, so there's no feedback loop to measure.")

Last season's minutes isn't in the chosen prior, so there's no feedback loop to measure.


## 6. The 2026-27 goalie projection

The chosen prior is refit on **all** target seasons and used for 2026-27. **League-level check** (as in notebook 18): the average goalie projection, weighted by 2025-26 minutes, with and without the prior. If it moves by more than 0.002 WAR/60, the prior is shifted by a constant so the league level stays where it was.

In [ ]:
mins = goalies[goalies["season"] == 2025].set_index("playerId")["toi"]
def level(p):
    p = p[p["played_last_season"]]
    return np.average(p["proj_war_60"], weights=mins.reindex(p.index))

cand26 = goalie_candidates(history, [2026]).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")

def final_projection(model, weight, cand=None):
    """2026-27 goalie projection with the prior at `weight` minutes, centered if the league level moves > 0.002."""
    pr = prior_rows(cand26 if cand is None else cand, model)
    proj = project(goalies, bios, 2026, G_WEIGHTS, G_REG, AGING, curve_full, priors=pr, prior_reg=weight)
    before, after = level(old_g), level(proj)
    shift = 0.0
    if abs(after - before) > 0.002:
        nudge = pr.assign(prior_gar_60=pr["prior_gar_60"] + 0.1)
        moved = level(project(goalies, bios, 2026, G_WEIGHTS, G_REG, AGING, curve_full, priors=nudge, prior_reg=weight))
        shift = -(after - before) / ((moved - after) / 0.1)
        pr["prior_gar_60"] += shift
        proj = project(goalies, bios, 2026, G_WEIGHTS, G_REG, AGING, curve_full, priors=pr, prior_reg=weight)
        assert abs(level(proj) - before) < 1e-6
    return proj, before, after, shift

shift = 0.0
if V:
    fit_all = goalie_fit_rows(history, feat, TARGETS)
    final = fit_prior(fit_all, models[V]["G"]["inputs"], fill_col=FILL.get(V, "pre_sv"))
    print(f"Final prior (version {V}), fit on {len(fit_all)} goalie-seasons:")
    print(model_table(final).round(4).to_string())
    new_g, before, after, shift = final_projection(final, W)
    print(f"\nLeague goalie level (WAR/60, weighted by 2025-26 minutes): without prior {before:.4f}, with prior {after:.4f} "
          f"(change {after - before:+.4f})")
    if shift:
        print(f"Centered: prior shifted by {shift:+.4f} GAR/60 ({shift / GOALS_PER_WIN:+.4f} WAR/60)")
    else:
        print("The goalie level moved by less than 0.002 WAR/60, so the prior is not shifted.")
else:
    new_g = old_g.copy()

Final prior (version d), fit on 969 goalie-seasons:
                 G        
              coef      se
constant    0.6675  0.0898
experience  0.0323  0.0062
age        -0.0186  0.0037
pre_sv      0.1067  0.0314

League goalie level (WAR/60, weighted by 2025-26 minutes): without prior 0.0485, with prior 0.0511 (change +0.0027)
Centered: prior shifted by -0.0239 GAR/60 (-0.0040 WAR/60)


## 7. OVR on the locked goalie scale

Goalie OVR with `ratings.ovr` (goalie anchors), with notebook 15's projection (before) and with the prior (after). "Regular" = 500+ minutes in 2025-26, as in notebook 20.

In [ ]:
g = old_g[["name", "team", "age_next", "played_last_season"]].rename(columns={"age_next": "age"})
g["old"], g["new"] = old_g["proj_war_60"], new_g["proj_war_60"].reindex(g.index)
g["prior"] = new_g["prior_gar_60"].reindex(g.index) / GOALS_PER_WIN
g["change"] = g["new"] - g["old"]
g["ovr_old"], g["ovr_new"] = ovr(g["old"], "G"), ovr(g["new"], "G")
g["minutes 2025-26"] = mins.reindex(g.index)
g = g.join(cand26.set_index("playerId")[["experience", "career_min", "last_min", "pre_sv", "pre_sv_raw"]])
reg = g[g["minutes 2025-26"] >= 500]
print(f"Regular goalies (500+ minutes in 2025-26): {len(reg)}")
summary = pd.DataFrame({
    "OVR min": [reg["ovr_old"].min(), reg["ovr_new"].min()], "OVR median": [reg["ovr_old"].median(), reg["ovr_new"].median()],
    "OVR max": [reg["ovr_old"].max(), reg["ovr_new"].max()],
    "90+": [(reg["ovr_old"] >= 90).sum(), (reg["ovr_new"] >= 90).sum()],
    "share 90+": [(reg["ovr_old"] >= 90).mean(), (reg["ovr_new"] >= 90).mean()]}, index=["before (notebook 15 projection)", "after"])
print(summary.round(3).to_string())
role = pd.cut(reg["minutes 2025-26"], [500, 1500, 2500, 5000], labels=["backup (500-1,500)", "split (1,500-2,500)", "starter (2,500+)"])
print("\nMedian OVR by 2025-26 role:")
print(reg.groupby(role, observed=True).agg(goalies=("ovr_new", "size"), before=("ovr_old", "median"), after=("ovr_new", "median")).to_string())

Regular goalies (500+ minutes in 2025-26): 72
                                 OVR min  OVR median  OVR max  90+  share 90+
before (notebook 15 projection)       75        80.0       91    2      0.028
after                                 72        80.0       95    3      0.042

Median OVR by 2025-26 role:
                     goalies  before  after
minutes 2025-26                            
backup (500-1,500)        20    79.0   79.0
split (1,500-2,500)       25    80.0   79.0
starter (2,500+)          27    82.0   83.0


In [ ]:
# Do the locked goalie anchors still match the data? (notebook 20 builds them from these regular goalies)
ws_reg = war_std(reg["new"], "G")
m_round = round(ws_reg.median() / 0.05) * 0.05
t_round = np.floor(war_std(g["new"], "G").max() / 0.05) * 0.05
print(f"Median regular goalie WAR_std: {ws_reg.median():.3f} (rounds to {m_round:.2f}; locked anchor {GOALIE_ANCHOR_WAR[2]:.2f})")
print(f"Top goalie WAR_std: {war_std(g['new'], 'G').max():.3f} (rounds down to {t_round:.2f}; locked anchor {GOALIE_ANCHOR_WAR[3]:.2f})")
if abs(m_round - GOALIE_ANCHOR_WAR[2]) > 1e-9 or abs(t_round - GOALIE_ANCHOR_WAR[3]) > 1e-9:
    print("The locked goalie anchors no longer match the data. ratings.py is NOT changed; notebook 20's assert will stop.")
else:
    print("The locked goalie anchors still match the data.")

Median regular goalie WAR_std: 2.278 (rounds to 2.30; locked anchor 2.30)
Top goalie WAR_std: 4.288 (rounds down to 4.25; locked anchor 4.25)
The locked goalie anchors still match the data.


## 8. Hockey checks

Starters with long track records should stay near the top; old backups and unproven goalies should fall.

In [ ]:
GSHOW = ["name", "team", "age", "experience", "career_min", "last_min", "pre_sv", "old", "prior", "new", "change", "ovr_old", "ovr_new"]
print("Top 15 goalies, new projection (old rank in brackets):")
top = g.sort_values("new", ascending=False).head(15).copy()
top["old rank"] = g["old"].rank(ascending=False).reindex(top.index).astype(int)
print(top[GSHOW + ["old rank"]].round(3).to_string(index=False))
played = g[g["played_last_season"]]
print("\nBiggest moves up (played in 2025-26):")
print(played.nlargest(15, "change")[GSHOW].round(3).to_string(index=False))
print("\nBiggest moves down:")
print(played.nsmallest(15, "change")[GSHOW].round(3).to_string(index=False))

Top 15 goalies, new projection (old rank in brackets):
               name team  age  experience  career_min  last_min  pre_sv   old  prior   new  change  ovr_old  ovr_new  old rank
     Logan Thompson  WSH 29.0           5   11903.017  3445.050   1.271 0.074  0.067 0.086   0.012       91       95         2
    Igor Shesterkin  NYR 31.0           7   18956.350  3024.217   0.649 0.068  0.060 0.076   0.008       88       91         3
  Connor Hellebuyck  WPG 33.0           7   36381.033  3401.450   0.292 0.075  0.048 0.076   0.001       91       91         1
        Carter Hart  VGK 28.0           7   13753.783  1019.783   1.096 0.044  0.078 0.070   0.026       80       89        77
     Jeremy Swayman  BOS 28.0           6   14196.650  3235.283   0.314 0.063  0.058 0.070   0.007       86       89         9
       Ilya Sorokin  NYI 31.0           6   17982.133  3226.167   0.649 0.063  0.055 0.068   0.005       86       88         8
 Andrei Vasilevskiy  TBL 32.0           7   35168.900  3

In [ ]:
WATCH = ["Cam Talbot", "Michael Hutchinson", "Sebastian Cossa", "Anthony Stolarz", "Thatcher Demko", "John Gibson",
         "Jakub Dobes", "Jet Greaves"]
print("Goalies to watch:")
print(g[g["name"].isin(WATCH)].sort_values("new", ascending=False)[GSHOW].round(3).to_string(index=False))
print("\nRed Wings goalies:")
print(g[g["team"] == "DET"].sort_values("new", ascending=False)[GSHOW].round(3).to_string(index=False))

Goalies to watch:
              name team  age  experience  career_min  last_min  pre_sv   old  prior   new  change  ovr_old  ovr_new
       Jet Greaves  CBJ 25.0           3    4511.017  3276.100  -0.157 0.067  0.043 0.065  -0.002       88       87
   Anthony Stolarz  TOR 33.0           7    8918.033  1389.250   0.150 0.065  0.045 0.065  -0.000       87       87
    Thatcher Demko  VAN 31.0           7   15224.917  1115.867   0.261 0.059  0.053 0.064   0.005       85       87
       Jakub Dobes  MTL 25.0           2    3400.367  2525.583   0.066 0.063  0.042 0.060  -0.003       87       85
       John Gibson  DET 33.0           7   31988.633  3181.233   0.472 0.051  0.051 0.054   0.003       82       83
   Sebastian Cossa  DET 24.0           0      45.000     0.000   0.538 0.047  0.042 0.043  -0.003       80       79
Michael Hutchinson  DET 36.0           6    8065.917     0.000   0.278 0.039  0.033 0.026  -0.013       78       76
        Cam Talbot  DET 39.0           7   32027.050  

**Is the tie rule safe?** The lowest-error weight and the chosen 12,000 minutes, compared season by season and by resampling the test goalie-seasons 2,000 times (each resample draws 370 goalie-seasons with replacement and recomputes both errors). If 12,000 comes out ahead in roughly half the resamples, the two are a tie and the choice rests on the reasons above. Then the top 10 under both: with a larger weight, a goalie's own last three seasons (at most about 10,000 minutes) count for less than his prior.

In [ ]:
if V and W != W_LOW:
    pr_test = prior_rows(cand_test, models[V])
    b_w, b_low = run(TEST, pr_test, W), run(TEST, pr_test, W_LOW)
    def rmse_of(b):
        return np.sqrt(np.average((b["actual"] - b["pred"]) ** 2, weights=b["toi"]))
    seasons = pd.DataFrame({f"{W:,} minutes": b_w.groupby("season").apply(rmse_of, include_groups=False),
                            f"{W_LOW:,} minutes": b_low.groupby("season").apply(rmse_of, include_groups=False)})
    seasons.index = [label(s) for s in seasons.index]
    print("Test error by season:")
    print(seasons.round(4).to_string())
    print(f"{W:,} minutes is better in {(seasons.iloc[:, 0] < seasons.iloc[:, 1]).sum()} of {len(seasons)} seasons")

    rng = np.random.default_rng(0)
    sq_w, sq_low, toi = ((b_w["actual"] - b_w["pred"]) ** 2 * b_w["toi"]).to_numpy(), ((b_low["actual"] - b_low["pred"]) ** 2 * b_low["toi"]).to_numpy(), b_w["toi"].to_numpy()
    diffs = []
    for _ in range(2000):
        i = rng.integers(0, len(toi), len(toi))
        diffs.append(np.sqrt(sq_w[i].sum() / toi[i].sum()) - np.sqrt(sq_low[i].sum() / toi[i].sum()))
    diffs = np.array(diffs)
    print(f"\nResampled error difference ({W:,} minus {W_LOW:,}): average {diffs.mean():+.4f}, 90% range {np.percentile(diffs, 5):+.4f} "
          f"to {np.percentile(diffs, 95):+.4f}; {W:,} minutes is better in {np.mean(diffs < 0):.0%} of resamples")

    alt = final_projection(final, W_LOW)[0]
    g["new_low"] = alt["proj_war_60"].reindex(g.index)
    g["ovr_low"] = ovr(g["new_low"], "G")
    side = g.sort_values("new", ascending=False).head(10)[["name", "team", "age", "new", "ovr_new"]].reset_index(drop=True)
    side_low = g.sort_values("new_low", ascending=False).head(10)[["name", "team", "age", "new_low", "ovr_low"]].reset_index(drop=True)
    print(f"\nTop 10 at {W:,} minutes, chosen (left), and at {W_LOW:,} minutes (right):")
    print(pd.concat([side, side_low], axis=1).round(3).to_string(index=False))
    reg_low = g[g["minutes 2025-26"] >= 500]
    print(f"\nRegular goalies at {W_LOW:,} minutes: OVR {reg_low['ovr_low'].min()}-{reg_low['ovr_low'].max()}, "
          f"median {reg_low['ovr_low'].median():.1f}, 90+ {(reg_low['ovr_low'] >= 90).sum()}")
else:
    print("The chosen weight is also the lowest-error weight, so there's nothing to compare.")

Test error by season:
         12,000 minutes  24,000 minutes
2018-19          0.2775          0.2745
2021-22          0.3146          0.3150
2022-23          0.3430          0.3437
2023-24          0.2676          0.2682
2024-25          0.2971          0.3003
2025-26          0.3024          0.2996
12,000 minutes is better in 4 of 6 seasons

Resampled error difference (12,000 minus 24,000): average -0.0000, 90% range -0.0023 to +0.0022; 12,000 minutes is better in 51% of resamples

Top 10 at 12,000 minutes, chosen (left), and at 24,000 minutes (right):
              name team  age   new  ovr_new              name team  age  new_low  ovr_low
    Logan Thompson  WSH 29.0 0.086       95    Logan Thompson  WSH 29.0    0.081       93
   Igor Shesterkin  NYR 31.0 0.076       91       Carter Hart  VGK 28.0    0.076       91
 Connor Hellebuyck  WPG 33.0 0.076       91   Igor Shesterkin  NYR 31.0    0.072       90
       Carter Hart  VGK 28.0 0.070       89    Jeremy Swayman  BOS 28.0    0.06

**Goalie prospects** get the rookie rule from section 4. For the prior, a prospect has no NHL experience, no NHL minutes and his own pre-NHL save%.

**Age: no unearned bonus.** In the prior, a younger goalie is better at the same experience. That comes from goalies who reached the NHL young **because they were good enough** to. A 19-year-old prospect is young because he's a prospect, which says nothing about whether he's good. So for goalie prospects only (no NHL experience), the prior is evaluated at **his age or the median age of a goalie's first 500+ minute NHL season, whichever is older**. That median is computed from the data (first seasons from 2008-09 on, MoneyPuck era). NHL goalies are unchanged.

In [ ]:
gp_ = pros[pros["group"] == "G"][["playerId", "name", "draft_team", "draft_year", "pick", "age", "league_2025_26",
                                  "gp_2025_26", "proj_war_60"]].rename(columns={"draft_team": "team", "proj_war_60": "old"})
gp_ = gp_.merge(pre_sv[["playerId", "pre_sv", "pre_sv_raw", "pre_sv_shots", "pre_sv_league", "pre_sv_gp"]], on="playerId", how="left")
gp_ = add_fades(gp_.assign(group="G", experience=0.0, career_min=0.0, log_career=0.0, last_min=0.0, last_k=0.0))
gp_["age_raw"] = gp_["age"]

# Median age at a goalie's first NHL season with 500+ minutes (2008-09 on), age on Feb 1 as everywhere else
debut = first_nhl[first_nhl >= 2008].rename("season").rename_axis("playerId").reset_index().assign(group="G")
debut = add_age(debut, bios)
DEBUT_AGE = float(debut["age"].median())
print(f"First 500+ minute NHL seasons since 2008-09: {len(debut)} goalies; age median {DEBUT_AGE:.0f} "
      f"(25th-75th percentile {debut['age'].quantile(0.25):.0f}-{debut['age'].quantile(0.75):.0f}; "
      f"through 2017-18 {debut.loc[debut['season'] <= TRAIN_END, 'age'].median():.0f}, after {debut.loc[debut['season'] > TRAIN_END, 'age'].median():.0f})")

# Before: the age floor only (the youngest fitted age). After: at least the median debut age.
gp_["age"] = gp_["age_raw"].clip(lower=AGE_FLOOR)
prior_floor_only = (apply_prior(gp_, final) + shift) / GOALS_PER_WIN if V else pd.Series(np.nan, index=gp_.index)
gp_["age"] = gp_["age_raw"].clip(lower=max(AGE_FLOOR, DEBUT_AGE))
print(f"Goalie prospects younger than {DEBUT_AGE:.0f}: {(gp_['age_raw'] < DEBUT_AGE).sum()} of {len(gp_)}, rated as {DEBUT_AGE:.0f}-year-olds")
prospect_prior = (apply_prior(gp_, final) + shift) / GOALS_PER_WIN if V else pd.Series(np.nan, index=gp_.index)
lg_avg_25 = group_average(history, 2025)["G"] / GOALS_PER_WIN
if ROOKIE_RULE.startswith("prior"):
    gp_["new"], gp_["rule"] = prospect_prior, f"goalie prior ({V})"
elif ROOKIE_RULE.startswith("league"):
    gp_["new"], gp_["rule"] = lg_avg_25, "league-average goalie"
else:
    gp_["new"], gp_["rule"] = 0.0, "replacement level"
gp_["prior_if_used"] = prospect_prior
gp_["ovr_old"], gp_["ovr_new"] = 70, ovr(gp_["new"], "G")      # notebook 20 gave goalie prospects a placeholder 70
print(f"Goalie prospects: {len(gp_)}, rule: {gp_['rule'].iloc[0]}; pre-NHL save% known for {gp_['pre_sv'].notna().mean():.0%}")
print(gp_["ovr_new"].describe()[["min", "25%", "50%", "75%", "max"]].to_string())
if V:
    fit_ages = fit_all["age"].dropna()
    print(f"\nAges in the prior's fitting data: youngest {fit_ages.min():.0f}, 5th percentile {fit_ages.quantile(0.05):.0f}, "
          f"median {fit_ages.median():.0f}. Each year younger adds {-final['G']['coef']['age'] / GOALS_PER_WIN:.4f} WAR/60 in the prior.")
PSHOW = ["name", "team", "draft_year", "pick", "age_raw", "league_2025_26", "gp_2025_26", "pre_sv_shots", "pre_sv_raw", "pre_sv",
         "pre_sv_league", "new", "ovr_new"]
print("\nTop 10 goalie prospects:")
print(gp_.nlargest(10, "new")[PSHOW].round(3).to_string(index=False))
print("\nRed Wings goalie prospects:")
print(gp_[gp_["team"] == "DET"].sort_values("new", ascending=False)[PSHOW].round(3).to_string(index=False))

First 500+ minute NHL seasons since 2008-09: 160 goalies; age median 24 (25th-75th percentile 22-25; through 2017-18 24, after 24)
Goalie prospects younger than 24: 120 of 139, rated as 24-year-olds
Goalie prospects: 139, rule: goalie prior (d); pre-NHL save% known for 96%
min    73.0
25%    76.0
50%    77.0
75%    78.0
max    81.0

Ages in the prior's fitting data: youngest 20, 5th percentile 22, median 28. Each year younger adds 0.0031 WAR/60 in the prior.

Top 10 goalie prospects:
             name team  draft_year  pick  age_raw league_2025_26  gp_2025_26  pre_sv_shots  pre_sv_raw  pre_sv pre_sv_league   new  ovr_new
  Semyon Vyazovoi  SEA        2021   163     23.0            KHL        53.0        2258.0       1.824   0.866           KHL 0.048       81
     Rudy Guimond  DET        2023   169     21.0          QMJHL        50.0        1924.0       1.973   0.858         QMJHL 0.048       81
    Dmitry Gamzin  NYI        2024   115     23.0            KHL        40.0        1551.0 

**Before vs. after the debut-age rule** for goalie prospects (before = the age floor of 20 only, as in the first runs of this notebook).

In [ ]:
da = pd.DataFrame({"name": gp_["name"], "team": gp_["team"], "age": gp_["age_raw"],
                   "before": prior_floor_only, "after": gp_["new"]})
da["OVR before"], da["OVR after"] = ovr(da["before"], "G"), ovr(da["after"], "G")
DA_SHOWN = ["Ryder Fetterolf", "Kim Saarinen", "Rudy Guimond", "Trey Augustine"]
print("Goalie prospects, WAR/60 and OVR (goalie scale):")
print(da[da["name"].isin(DA_SHOWN)].set_index("name").reindex(DA_SHOWN).round(4).to_string())
for col in ["before", "after"]:
    o = da[f"OVR {col}"]
    print(f"{col:>6}: WAR/60 median {da[col].median():.4f} (range {da[col].min():.4f} to {da[col].max():.4f}); "
          f"OVR median {o.median():.0f} (range {o.min()}-{o.max()})")
print(f"Change for prospects younger than {DEBUT_AGE:.0f}: {(da['after'] - da['before'])[da['age'] < DEBUT_AGE].mean():+.4f} WAR/60 on average; "
      f"{DEBUT_AGE:.0f} and older: unchanged ({(da['after'] - da['before'])[da['age'] >= DEBUT_AGE].abs().max():.4f})")

Goalie prospects, WAR/60 and OVR (goalie scale):
                team   age  before   after  OVR before  OVR after
name                                                             
Ryder Fetterolf  CAR  19.0  0.0580  0.0456          85         80
Kim Saarinen     SEA  20.0  0.0582  0.0457          85         80
Rudy Guimond     DET  21.0  0.0574  0.0481          84         81
Trey Augustine   DET  21.0  0.0516  0.0423          82         79
before: WAR/60 median 0.0418 (range 0.0194 to 0.0582); OVR median 79 (range 74-85)
 after: WAR/60 median 0.0325 (range 0.0128 to 0.0482); OVR median 77 (range 73-81)
Change for prospects younger than 24: -0.0097 WAR/60 on average; 24 and older: unchanged (0.0000)


**Before vs. after the two save% and age safeguards.** "Before" is this notebook's first setup, rebuilt here so the comparison holds on any rerun: raw pre-NHL save% (no shrinking), no age floor, version (a) + pre-NHL save% at 12,000 minutes, centered the same way. "After" is the final projection, including the debut-age rule for prospects.

In [ ]:
feat_b = goalie_features(pairs, history, bios, profiles, gseasons, first_nhl)        # raw save%, no age floor
fit_b = goalie_fit_rows(history, feat_b, TARGETS)
final_b = fit_prior(fit_b, GOALIE_VERSIONS["a"] + ["pre_sv"], fill_col="pre_sv")
cand26_b = goalie_candidates(history, [2026]).merge(feat_b.drop(columns="group"), on=["playerId", "season"], how="left")
new_b, _, _, shift_b = final_projection(final_b, 12000, cand26_b)
pre_b = pre_nhl_save_pct(gseasons, first_nhl, before=2026)
gp_b = gp_[["playerId", "name", "age_raw"]].rename(columns={"age_raw": "age"}).merge(pre_b[["playerId", "pre_sv"]], on="playerId", how="left")
gp_b = gp_b.assign(group="G", experience=0.0)
prior_b = pd.Series(((apply_prior(gp_b, final_b) + shift_b) / GOALS_PER_WIN).to_numpy(), index=gp_b["playerId"])

cmp = pd.concat([
    pd.DataFrame({"name": g["name"], "before": new_b["proj_war_60"].reindex(g.index), "after": g["new"]}),
    pd.DataFrame({"name": gp_["name"].to_numpy(), "before": prior_b.to_numpy(), "after": gp_["new"].to_numpy()}, index=gp_["playerId"])])
cmp["OVR before"], cmp["OVR after"] = ovr(cmp["before"], "G"), ovr(cmp["after"], "G")
SHOWN = ["Logan Thompson", "Carter Hart", "Connor Hellebuyck", "Ryder Fetterolf", "Matvei Nikonovich"]
print("Before vs. after (WAR/60 and OVR):")
print(cmp[cmp["name"].isin(SHOWN)].set_index("name").reindex(SHOWN).round(3).to_string())
pb = cmp.loc[gp_["playerId"]]
print(f"\nGoalie prospects' median OVR: before {pb['OVR before'].median():.0f}, after {pb['OVR after'].median():.0f} "
      f"(range before {pb['OVR before'].min()}-{pb['OVR before'].max()}, after {pb['OVR after'].min()}-{pb['OVR after'].max()})")
rb = cmp.loc[reg.index]
print(f"Regular goalies: OVR before {rb['OVR before'].min()}-{rb['OVR before'].max()} (median {rb['OVR before'].median():.0f}, "
      f"90+ {(rb['OVR before'] >= 90).sum()}), after {rb['OVR after'].min()}-{rb['OVR after'].max()} "
      f"(median {rb['OVR after'].median():.0f}, 90+ {(rb['OVR after'] >= 90).sum()})")

Before vs. after (WAR/60 and OVR):
                   before  after  OVR before  OVR after
name                                                   
Logan Thompson      0.087  0.086          96         95
Carter Hart         0.069  0.070          89         89
Connor Hellebuyck   0.076  0.076          91         91
Ryder Fetterolf     0.067  0.046          88         80
Matvei Nikonovich   0.065  0.043          87         79

Goalie prospects' median OVR: before 79, after 77 (range before 72-88, after 73-81)
Regular goalies: OVR before 71-96 (median 80, 90+ 3), after 72-95 (median 80, 90+ 3)


## 9. Is goalie WAR on the same scale as skater WAR?

Two tests.

**Test 1, replacement level.** WAR is measured above "freely available" players. If the replacement level is right, freely available players should average about 0 WAR/60. Pools:

- goalies outside each team's **top 2 by minutes** and skaters outside each team's **top 13 forwards / 7 defensemen** (these pools *define* replacement in notebooks 06 and 14, so their overall average is near 0 by construction; the independent evidence is whether it holds in every period),
- **first-time call-ups:** goalies and skaters in their first NHL season in our data (2009-10 on, nobody with earlier NHL games), all minutes. These are a pool the WAR model never used to set its levels.

Minutes-weighted WAR/60 with standard errors, by period.

In [ ]:
gw = goalie_war.copy()
sw = skater_war.copy()
sw["rank"] = sw.groupby(["season", "team", "pos_group"])["all_toi"].rank(ascending=False, method="first")
sw["outside"] = np.where(sw["pos_group"] == "F", sw["rank"] > 13, sw["rank"] > 7)
early_nhl = set(profiles[(profiles["league"] == "NHL") & (profiles["season"] < 2008)]["playerId"])
first_g = gw.groupby("playerId")["season"].transform("min")
first_s = sw.groupby("playerId")["season"].transform("min")
gw["callup"] = (gw["season"] == first_g) & (gw["season"] >= 2009) & ~gw["playerId"].isin(early_nhl)
sw["callup"] = (sw["season"] == first_s) & (sw["season"] >= 2009) & ~sw["playerId"].isin(early_nhl)

PERIODS = {"2008-09 to 2013-14": range(2008, 2014), "2014-15 to 2019-20": range(2014, 2020), "2020-21 to 2025-26": range(2020, 2026)}
def pool_rate(d, toi_col):
    """Minutes-weighted WAR/60 and its standard error (each player-season is one observation)."""
    x, w = d["war"] / d[toi_col] * 60, d[toi_col]
    m = np.average(x, weights=w)
    se = np.sqrt(np.sum(w ** 2 * (x - m) ** 2)) / w.sum()
    return pd.Series({"player-seasons": len(d), "minutes": w.sum(), "WAR/60": m, "SE": se})

rows = []
for pname, seasons in list(PERIODS.items()) + [("all seasons", range(2008, 2026))]:
    for pool, d, col in [("goalies outside team top 2", gw[~gw["is_starter"]], "toi"),
                         ("goalie first-time call-ups", gw[gw["callup"]], "toi"),
                         ("skaters outside top 13 F / 7 D", sw[sw["outside"]], "all_toi"),
                         ("skater first-time call-ups", sw[sw["callup"]], "all_toi")]:
        d = d[d["season"].isin(seasons) & (d[col] > 0)]
        rows.append({"period": pname, "pool": pool, **pool_rate(d, col)})
repl = pd.DataFrame(rows)
print(repl.round(4).to_string(index=False))

            period                           pool  player-seasons     minutes  WAR/60     SE
2008-09 to 2013-14     goalies outside team top 2           168.0  51314.3500  0.0064 0.0084
2008-09 to 2013-14     goalie first-time call-ups            72.0  33623.2000  0.0435 0.0115
2008-09 to 2013-14 skaters outside top 13 F / 7 D          1674.0 242812.9667 -0.0053 0.0023
2008-09 to 2013-14     skater first-time call-ups           626.0 175602.1333  0.0273 0.0032
2014-15 to 2019-20     goalies outside team top 2           186.0  57987.1000 -0.0141 0.0102
2014-15 to 2019-20     goalie first-time call-ups            82.0  32956.5000  0.0331 0.0131
2014-15 to 2019-20 skaters outside top 13 F / 7 D          1686.0 264314.5167 -0.0035 0.0021
2014-15 to 2019-20     skater first-time call-ups           797.0 238153.9833  0.0279 0.0032
2020-21 to 2025-26     goalies outside team top 2           241.0  82991.1500  0.0059 0.0081
2020-21 to 2025-26     goalie first-time call-ups            90.0  320

**Test 2, team goal differential.** If one goalie WAR and one skater WAR are worth the same, a team's goal differential should rise by the same amount for each. Regression of **team goal differential per 82 games** on team skater WAR and team goalie WAR as separate terms, 2008-09 to 2025-26 (the three short seasons are skipped). Standings points aren't in our data and this notebook makes no network calls, so goal differential stands in: about 6 goals = 1 win, which is how WAR is built.

- Goals for = the team's skaters' goals (all situations); goals against = its goalies' goals against (all situations; empty-net goals aren't charged to a goalie). A traded player counts for his last team, as in notebooks 06 and 14.
- Each team plays about the same goalie minutes, so this tests the **size** of a goalie WAR (is a goalie win really a win?), not its replacement level. Test 1 covers the level.
- If the goalie coefficient is within its standard error of the skater one, goalie WAR is on the same scale. If it's clearly smaller, skater coefficient / goalie coefficient is the implied inflation factor.

In [ ]:
sk_raw = pd.concat([pd.read_csv(DATA / "skaters_2008_to_2024.csv"), pd.read_csv(DATA / "skaters25-26.csv")], ignore_index=True)
g_raw = pd.concat([pd.read_csv(DATA / "goalies_2008_to_2024.csv"), pd.read_csv(DATA / "goalies25-26.csv")], ignore_index=True)
gf = sk_raw[sk_raw["situation"] == "all"].groupby(["season", "team"])["I_F_goals"].sum()
ga = g_raw[g_raw["situation"] == "all"].groupby(["season", "team"])["goals"].sum()
teams = pd.DataFrame({"gf": gf, "ga": ga,
                      "skater_war": sw.groupby(["season", "team"])["war"].sum(),
                      "goalie_war": gw.groupby(["season", "team"])["war"].sum()}).dropna()
teams = teams[~teams.index.get_level_values("season").isin(SHORT_SEASONS)]
teams["gd"] = teams["gf"] - teams["ga"]                    # full seasons are 82 games, so this is already per 82
X = pd.DataFrame({"constant": 1.0, "skater_war": teams["skater_war"], "goalie_war": teams["goalie_war"]})
beta, se, r2 = wls(teams["gd"], X, np.ones(len(teams)))
team_fit = pd.DataFrame({"coef (goals per WAR)": beta, "se": se}, index=X.columns)
print(f"Team-seasons: {len(teams)} ({teams.index.get_level_values('season').nunique()} full seasons); R^2 {r2:.3f}")
print(team_fit.round(3).to_string())
b_s, b_g, se_s, se_g = beta[1], beta[2], se[1], se[2]
diff_se = np.sqrt(se_s ** 2 + se_g ** 2)
print(f"\nGoals per win used to build WAR: {GOALS_PER_WIN:.2f}")
print(f"Goalie minus skater coefficient: {b_g - b_s:+.2f} (SE of the difference {diff_se:.2f}, z {(b_g - b_s) / diff_se:+.1f})")
print(f"Skater / goalie coefficient: {b_s / b_g:.2f}")

Team-seasons: 462 (15 full seasons); R^2 0.552
            coef (goals per WAR)     se
constant                 -74.112  3.979
skater_war                 4.663  0.264
goalie_war                 6.168  0.540

Goals per win used to build WAR: 6.00
Goalie minus skater coefficient: +1.51 (SE of the difference 0.60, z +2.5)
Skater / goalie coefficient: 0.76


In [ ]:
# Plain verdict from the two tests
call_g = repl[(repl["period"] == "all seasons") & (repl["pool"] == "goalie first-time call-ups")].iloc[0]
call_s = repl[(repl["period"] == "all seasons") & (repl["pool"] == "skater first-time call-ups")].iloc[0]
gap = call_g["WAR/60"] - call_s["WAR/60"]
gap_se = np.sqrt(call_g["SE"] ** 2 + call_s["SE"] ** 2)
print(f"Test 1: first-time call-ups average {call_g['WAR/60']:+.4f} WAR/60 for goalies (SE {call_g['SE']:.4f}) "
      f"and {call_s['WAR/60']:+.4f} for skaters (SE {call_s['SE']:.4f}); difference {gap:+.4f} (SE {gap_se:.4f}).")
print(f"Test 2: a goalie WAR is worth {b_g:.2f} goals of goal differential (SE {se_g:.2f}), a skater WAR {b_s:.2f} (SE {se_s:.2f}).")
level_ok = abs(gap) <= 2 * gap_se
if b_g < b_s - diff_se:
    verdict = f"Goalie WAR is inflated by about {b_s / b_g:.2f}x relative to skater WAR."
elif level_ok:
    verdict = ("Goalies are on the same scale; the cluster is the job's value. A goalie WAR buys at least as much "
               "goal differential as a skater WAR, and goalie call-ups sit as far above replacement as skater call-ups.")
else:
    verdict = (f"A goalie WAR is not inflated in size, but goalie call-ups sit {gap:+.4f} WAR/60 away from skater "
               "call-ups, so the two replacement levels differ.")
print("\nVERDICT: " + verdict)

Test 1: first-time call-ups average +0.0297 WAR/60 for goalies (SE 0.0071) and +0.0256 for skaters (SE 0.0019); difference +0.0041 (SE 0.0073).
Test 2: a goalie WAR is worth 6.17 goals of goal differential (SE 0.54), a skater WAR 4.66 (SE 0.26).

VERDICT: Goalies are on the same scale; the cluster is the job's value. A goalie WAR buys at least as much goal differential as a skater WAR, and goalie call-ups sit as far above replacement as skater call-ups.


**Reading the two tests honestly:**

- **Call-ups land above 0 for both positions.** First-time call-ups are better than the "outside the top 13/7 or top 2" pool (they're often promising players getting a chance), so neither replacement level is too *high*. What matters for one scale is that goalies and skaters sit about the same distance above replacement.
- **The goalie coefficient is close to 6 partly by construction.** Goalie WAR is goals saved compared with expected goals, and goals against are exactly what goal differential counts, so most of it flows straight through. The skater coefficient sits **below** 6, which says a skater WAR buys a little less goal differential than it claims. Part of that is noise in the skater measures (noise in an input always pulls its coefficient toward 0), part may be that skater WAR is a bit generous. Either way it points the opposite way from "goalie WAR is inflated".

**Scale options** (no change is made here; this is a decision for Ryan):

1. **Keep the scale.** If goalie WAR is on the same scale, a starter really is worth more than almost any skater over a season, and high goalie OVRs are the honest reading of "one scale". The goalie prior still sorts goalies better.
2. **Goalie workload.** OVR uses a starter's ~50 games. A smaller standard workload (for example, the games where a starter vs. his backup actually changes the result) would lower every goalie's WAR_std. This changes what "a 90 goalie" means, so it needs a hockey argument, not just a fit.
3. **Goalie replacement level** (notebook 14): if the first-time call-up goalies sit clearly above 0, the replacement level is too low and every goalie WAR is too high by the same amount per minute. Raising it would move all goalies down together.
4. **A goalie-specific OVR mapping.** Rate goalies against goalies (for example, median regular goalie = 80). Simple and EA-like, but it breaks "a 90 at any position is worth the same to a team".

The verdict above says which of these the data supports.

## 10. Save

`data/goalie_projections_2026-27.csv`: every NHL goalie in the 2026-27 projection (old = notebook 15/18, prior, new) and every goalie prospect (old = notebook 19's league-average goalie, new = the rookie rule), with the inputs and OVR before and after. `projections_2026-27.csv` stays as notebook 18 left it.

In [ ]:
nhl_rows = g.rename_axis("playerId").reset_index().assign(prospect=False, rule=f"goalie prior ({V})" if V else "notebook 15")
nhl_rows = nhl_rows.rename(columns={"old": "old_proj_war_60", "new": "new_proj_war_60", "prior": "prior_war_60",
                                    "minutes 2025-26": "minutes_2025_26", "played_last_season": "played_2025_26"})
p_rows = gp_.drop(columns="age").rename(columns={"age_raw": "age", "old": "old_proj_war_60", "new": "new_proj_war_60",
                                                 "prior_if_used": "prior_war_60"}).assign(prospect=True, played_2025_26=False)
COLS = ["playerId", "name", "team", "age", "prospect", "played_2025_26", "minutes_2025_26", "experience", "career_min",
        "last_min", "pre_sv_raw", "pre_sv", "old_proj_war_60", "prior_war_60", "new_proj_war_60", "ovr_old", "ovr_new", "rule"]
out = pd.concat([nhl_rows.reindex(columns=COLS), p_rows.reindex(columns=COLS)], ignore_index=True)
assert out["playerId"].is_unique and out["new_proj_war_60"].notna().all()
out.to_csv(OUT / "goalie_projections_2026-27.csv", index=False)
print(f"Saved goalie_projections_2026-27.csv: {len(out)} goalies ({(~out['prospect']).sum()} NHL, {out['prospect'].sum()} prospects)")

Saved goalie_projections_2026-27.csv: 271 goalies (132 NHL, 139 prospects)


**One place to read projections (done).** `forecast.current_projections()` collects every 2026-27 projection: notebook 18's skaters, this notebook's NHL goalies and goalie prospects, and notebook 19's skater prospects. Notebook 20 (second version) rates players from it and writes the app's forecast table, so after a rerun of this notebook, rerun notebook 20 to refresh the ratings.

## Takeaways
- **The goalie prior helps, a little.**
  - **Error:** on 370 test goalie-seasons (2018-19 on, prior fit through 2017-18), the chosen version cuts the error from 0.3043 to 0.3015 GAR/60 (0.9%):
    - 3 or fewer seasons: 0.3177 to 0.3134
    - veterans: 0.2963 to 0.2944
  - **The chosen version:** experience + age + pre-NHL save% (version d, save% shrunk by sample size), at 12,000 minutes of weight.
  - **500+ minute goalies (more backups):** 0.3196 to 0.3169.
  - **Small gains are expected:** goalie results barely repeat (notebook 15).
- **Pre-NHL save% is shrunk by sample size, with K estimated from the data.**
  - **How much it repeats:** between a goalie's consecutive pre-NHL seasons (1,888 pairs), relative save% correlates only 0.19. The spread of true talent is 0.56 points of save%.
  - **The estimate:** K = 2,591 shots, rounded to 2,500. A record counts 17% at 500 shots, 29% at 1,000, 44% at 2,000, 62% at 4,000. The median goalie's last two pre-NHL seasons hold 1,812 shots, and shrinking cuts the spread of pre-NHL save% from 1.03 to 0.40 points.
  - **Missing shots:** 3.3% of seasons are filled in as games x league shots per game.
  - **The test seasons don't favor shrinking:** (d) at 12,000 minutes scores 0.3011 raw vs. 0.3015 at K = 2,500 (1,000: 0.3013; 4,000: 0.3016). The gaps are noise (the resampling below puts the 90% range at about ±0.0023). K was taken from the repeat estimate, not picked on the test seasons.
- **Age floor:** the youngest goalie in the fitting data is 20, so younger goalies count as 20. That affects 4 NHL goalie-seasons and 39 of 139 prospects.
- **Fading pre-NHL save% isn't used.**
  - Fading it to 0 at 7 NHL seasons (e2) was 0.0001 better than (d) (0.3013 vs. 0.3014 at 24,000 minutes).
  - Counting it only for 3 or fewer seasons (e1) didn't win.
  - A fading version must beat (d) by more than 0.001 to replace it, the same threshold as the weight tie rule. (d) is simpler, and it's better on goalies with 500+ minutes (0.3169 vs. 0.3176) and on goalies with 3 or fewer seasons (0.3134 vs. 0.3146).
  - Under (e2), goalies with almost no NHL record jumped into the top 8 (DiPietro, with 132 career NHL minutes).
- **12,000 minutes, by the tie rule.** 24,000 had a 0.0001 lower error, but 12,000 is better in 4 of 6 test seasons and in 51% of 2,000 resamples.
- **What predicts a goalie** (training fit, 527 goalie-seasons, version d):
  - Each season of NHL experience adds 0.029 GAR/60 (z 3.6), and each year of age takes away 0.018 (z -3.9).
  - Each shrunk point of pre-NHL save% adds 0.116 (z 2.5).
  - Career minutes and last season's minutes add nothing once experience is in, so there is **no minutes feedback loop** for the simulator.
- **Rookie goalies are close to league average, not replacement.**
  - 54 first 500+ minute seasons since 2018-19 averaged +0.041 WAR/60.
  - The prior alone (0.4398 GAR/60 error) beats the league average (0.4440) and replacement (0.5051).
  - So goalie prospects get the prior, with the debut-age rule below.
- **Goalie prospects don't get an unearned age bonus.**
  - In the prior a younger goalie is better at the same experience, but that comes from goalies who reached the NHL young because they were good. A 19-year-old prospect is young because he's a prospect.
  - **The rule:** goalie prospects (no NHL experience) are rated at their own age or the median age of a goalie's first 500+ minute NHL season, whichever is older. NHL goalies are unchanged.
  - **The median debut age is 24** (160 goalies since 2008-09; 25th-75th percentile 22-25; 24 both through 2017-18 and after). 120 of 139 prospects are younger, and they lose 0.0097 WAR/60 on average.
  - **Prospect projections:** median 0.0418 to 0.0325 WAR/60. That's now below what true rookie goalies averaged (+0.041), which fits: a rookie had already earned his 500 minutes.

  | | Before | After |
  |---|---|---|
  | Fetterolf (19) | 85 | 80 |
  | Saarinen (20) | 85 | 80 |
  | Guimond (21) | 84 | 81 |
  | Augustine (21) | 82 | 79 |
  | Goalie-prospect median (range) | 79 (74-85) | 77 (73-81) |

- **2026-27:**
  - The final prior raises the goalie level by 0.0027 WAR/60, so it's shifted down by 0.0040 WAR/60.
  - The locked goalie OVR anchors still match the data: the median regular goalie is 2.278 WAR_std (rounds to 2.30), and the top goalie is 4.288 (rounds down to 4.25).
- **OVR in this notebook is on the goalie scale.** After the first runs, Ryan gave goalies their own anchors in notebook 20 (median regular goalie = 80, top goalie = 95). The projections don't depend on that; only the OVR columns changed.
- **Before vs. after all the safeguards** (before = raw save%, no age floor, no debut-age rule, same version and weight):

  | | Before | After |
  |---|---|---|
  | Thompson | 96 | 95 |
  | Hart | 89 | 89 |
  | Hellebuyck | 91 | 91 |
  | Fetterolf | 88 | 80 |
  | Nikonovich | 87 | 79 |
  | Goalie-prospect median | 79 | 77 |

  The prospect range narrows from 72-88 to 73-81. Shrinking save% pulls both ends toward the middle, and the debut-age rule takes away the youngest prospects' bonus.
- **The prior spreads goalies by role.**
  - **Regular goalies:** 75-91 with notebook 15's projection, 72-95 with the prior. The median is 80 both ways (by construction of the goalie anchors), and 90+ goes from 2 to 3 of 72.
  - **Median by 2025-26 role:** starters 82 to 83, splits 80 to 79, backups 79 both.
  - **Old backups and goalies with weak pre-NHL numbers fall:** Shepard 79 to 70, Copley 78 to 71, Quick (41) 77 to 72, Talbot 78 to 75, Hutchinson 78 to 76.
- **The top: Thompson 95, Shesterkin and Hellebuyck 91, Hart and Swayman 89.**
  - **Flag:** Hart (up from 80) rides his age, seven seasons and strong WHL save% on only 1,020 NHL minutes last season.
- **Goalie prospects:** median 77 (73 to 81); Vyazovoi and Guimond are top at 81. Like skater prospects, a rating means "if he plays", and readiness belongs in Phase 2's call-up logic.
- **Red Wings:** Gibson 83, Cossa 79, Hutchinson 76, Talbot 75.
  - Goalie prospects: Guimond 81, Augustine and Bednar 79, Pradel 77, Orsulak 76, Miller 74.
- **Is goalie WAR on the same scale?** Yes.
  - **Test 1:** first-time call-ups average +0.030 WAR/60 for goalies (SE 0.007) and +0.026 for skaters (SE 0.002), so both sit the same distance above replacement.
  - **Test 2:** across 462 team-seasons, a goalie WAR is worth 6.17 goals of goal differential (SE 0.54) and a skater WAR 4.66 (SE 0.26).
  - **Verdict:** goalies are on the same scale; the cluster is the job's value.
- **For the Phase 2 simulator: a skater WAR buys 4.66 goals of goal differential, not the 6.0 used to build WAR.**
  - So the simulator should turn team WAR into goals with **fitted coefficients**: about 4.7 goals per skater WAR and 6.2 per goalie WAR, refit on the simulator's own setup, instead of 6.0 for both.
  - Using 6.0 would overstate how much better a team with more skater WAR is.
- **The scale decision (Ryan, notebook 20's second version):** WAR stays on one scale; OVR gets goalie-specific anchors. The simulator uses WAR, and goalie projections are far less certain, so the best goalie shows at 95, just below the best skaters.
- **Files:** `goalie_projections_2026-27.csv` holds 132 NHL goalies and 139 goalie prospects, with raw and shrunk pre-NHL save% (OVR columns on the goalie scale). `projections_2026-27.csv` is unchanged. After rerunning this notebook, rerun notebook 20 to refresh the ratings and the app's forecast table.
- **Next:** minutes projections and the season simulator.